# WSDM 2027 — Experiment 1 full 41-unit run

Notebook này chạy **proposed method** trên 41 argument units thuộc điều kiện `human_segmented`. Đây là approved pilot có development overlap, không phải held-out evaluation hay bằng chứng generalization.

Trình tự bắt buộc: (1) GPU, source và input audit; (2) model setup; (3) inference 41 units với atomic checkpoints; (4) lưu inference output; (5) chỉ sau đó attach/read gold và evaluate. Không truyền gold, annotation explanations hoặc evaluation manifest vào model.

In [ ]:
import json, os, subprocess
gpu_query = [
    "nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.free,compute_cap",
    "--format=csv,noheader,nounits",
]
gpu_result = subprocess.run(gpu_query, text=True, capture_output=True, check=True)
print(gpu_result.stdout)
GPU_REPORT = []
for line in gpu_result.stdout.strip().splitlines():
    index, name, driver, total, free, capability = [x.strip() for x in line.split(",")]
    GPU_REPORT.append({
        "index": int(index), "name": name, "driver_version": driver,
        "memory_total_mib": int(total), "memory_free_mib": int(free),
        "compute_capability": capability,
    })
assert GPU_REPORT and GPU_REPORT[0]["memory_free_mib"] >= 10000, GPU_REPORT
print("GPU checkpoint: PASS")

## Checkpoint 1 — source và full input audit

Notebook ghim riêng implementation commit của runner để tránh self-pin vòng tròn. Nó lấy HEAD hiện tại của remote branch làm execution commit, xác minh implementation là ancestor, runner không đổi kể từ implementation pin, và `llm.py`/`steering.py` không khác frozen diagnostic commit. Run manifest ghi chính xác execution commit. Attach **chỉ** dataset input chứa `model_inputs_human_segmented.csv`; chưa attach gold ở checkpoint này.

In [ ]:
from pathlib import Path
import hashlib, inspect, shutil, sys
import pandas as pd

REPO_URL = "https://github.com/dlxik/vip-girls.git"
SOURCE_BRANCH = "data/cmv-edge-review-20261116"
FROZEN_METHOD_COMMIT = "02f97218e5726588bc4c29fbdb61f49bbed28127"
PINNED_IMPLEMENTATION_COMMIT = "d2f2b1279280a6514482d64467de0c8591bfdf96"

INPUT_FILENAME = "model_inputs_human_segmented.csv"
INPUT_SHA256 = "52839A7E92C249FAEECAE491CB22167235A2F767E0E31486036468875AB466B1"
EXPECTED_ITEMS = 41
REPO_DIR = Path("/kaggle/working/vip-girls")
WORKING_INPUT = Path("/kaggle/working") / INPUT_FILENAME

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], check=True)
EXECUTION_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", f"origin/{SOURCE_BRANCH}"], text=True
).strip()
subprocess.run([
    "git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor",
    PINNED_IMPLEMENTATION_COMMIT, EXECUTION_COMMIT,
], check=True)
# Refuse execution if the actual runner changed after the reviewed implementation pin.
subprocess.run([
    "git", "-C", str(REPO_DIR), "diff", "--exit-code",
    PINNED_IMPLEMENTATION_COMMIT, EXECUTION_COMMIT, "--",
    "wsdm/approved_detection_experiment.py",
], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", EXECUTION_COMMIT], check=True)
resolved_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert resolved_commit == EXECUTION_COMMIT
# Freeze method and formula files at the reviewed diagnostic implementation.
subprocess.run([
    "git", "-C", str(REPO_DIR), "diff", "--exit-code",
    FROZEN_METHOD_COMMIT, EXECUTION_COMMIT, "--", "wsdm/llm.py", "wsdm/steering.py",
], check=True)

matches = sorted(Path("/kaggle/input").rglob(INPUT_FILENAME))
if len(matches) != 1:
    raise RuntimeError(f"Cần đúng 1 full input file; tìm thấy: {matches}")
attached_input = matches[0]
input_sha256 = hashlib.sha256(attached_input.read_bytes()).hexdigest().upper()
assert input_sha256 == INPUT_SHA256, (input_sha256, INPUT_SHA256)
shutil.copy2(attached_input, WORKING_INPUT)

WSDM_DIR = REPO_DIR / "wsdm"
sys.path.insert(0, str(WSDM_DIR))
import approved_detection_experiment as approved_exp
inputs = approved_exp._read_csv(WORKING_INPUT)
approved_exp._assert_model_input(inputs)
assert len(inputs) == EXPECTED_ITEMS
assert inputs.argument_pair_id.nunique() == EXPECTED_ITEMS
assert set(inputs.input_condition) == {"human_segmented"}
assert inputs.human_segmentation_used.astype(str).str.upper().eq("TRUE").all()
forbidden = [c for c in inputs.columns if any(
    token in c.lower() for token in ("label", "steering", "evidence", "adjudication", "gold")
)]
assert forbidden == [], forbidden
print(f"Pinned implementation commit: {PINNED_IMPLEMENTATION_COMMIT}")
print(f"Exact execution commit: {resolved_commit}")
print(f"Frozen method commit: {FROZEN_METHOD_COMMIT}")
print(f"Input SHA-256: {input_sha256}; rows={len(inputs)}; unique IDs={inputs.argument_pair_id.nunique()}")
print("Gold isolation and full-input checkpoint: PASS")

## Checkpoint 2 — Ollama + frozen Qwen model

Giữ nguyên Qwen2.5-7B-Instruct Q4_K_M, context 8192, temperature 0 và seed 0. Cell tải đúng hai GGUF shards chính thức, merge và import dưới model ID `qwen2.5:7b`.

In [ ]:
import time, urllib.request
MODEL_ID = "qwen2.5:7b"
MODEL_SPEC = f"ollama:{MODEL_ID}"
NUM_CTX, TEMPERATURE, SEED, MAX_RETRIES = 8192, 0, 0, 1
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
OLLAMA_LOG = Path("/kaggle/working/ollama_server.log")
def ollama_ready():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3) as response:
            return response.status == 200
    except Exception:
        return False
if not ollama_ready():
    log_handle = OLLAMA_LOG.open("a", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"], stdout=log_handle, stderr=subprocess.STDOUT, env=os.environ.copy()
    )
    for _ in range(60):
        if ollama_ready(): break
        time.sleep(1)
    else: raise RuntimeError("Ollama server không sẵn sàng")

HF_BASE = "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main"
MODEL_DIR = Path("/kaggle/working/qwen2.5-7b-instruct-q4_k_m")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SHARDS = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]
for name in SHARDS:
    path = MODEL_DIR / name
    if not path.exists():
        subprocess.run([
            "curl", "-fL", "--retry", "5", "--retry-all-errors",
            "-o", str(path), f"{HF_BASE}/{name}?download=true",
        ], check=True)
    assert path.stat().st_size > 100_000_000, path
LLAMA_CPP = Path("/kaggle/working/llama.cpp")
if not LLAMA_CPP.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP)], check=True)
subprocess.run(["cmake", "-S", str(LLAMA_CPP), "-B", str(LLAMA_CPP / "build"), "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF"], check=True)
subprocess.run(["cmake", "--build", str(LLAMA_CPP / "build"), "--target", "llama-gguf-split", "-j", "2"], check=True)
MERGED = MODEL_DIR / "qwen2.5-7b-instruct-q4_k_m.gguf"
if not MERGED.exists():
    subprocess.run([str(LLAMA_CPP / "build/bin/llama-gguf-split"), "--merge", str(MODEL_DIR / SHARDS[0]), str(MERGED)], check=True)
assert MERGED.stat().st_size > 4_000_000_000
MODELFILE = MODEL_DIR / "Modelfile"
MODELFILE.write_text(f"FROM {MERGED}\n", encoding="utf-8")
subprocess.run(["ollama", "create", MODEL_ID, "-f", str(MODELFILE)], check=True)
with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10) as response:
    model_records = json.loads(response.read()).get("models", [])
MODEL_METADATA = next(x for x in model_records if x.get("name") in {MODEL_ID, f"{MODEL_ID}:latest"})
MODEL_DIGEST = MODEL_METADATA["digest"]
assert MODEL_METADATA["details"]["quantization_level"] == "Q4_K_M"
print(json.dumps(MODEL_METADATA, indent=2, ensure_ascii=False))
print("Model checkpoint: PASS")

## Checkpoint 3 — inference 41 units, checkpoint/resume

Cell này không nhận gold path. Mỗi processed item tạo một atomic checkpoint. Chạy lại cell với cùng output/config sẽ skip mọi completed item. Nếu tiếp tục ở session mới, attach một dataset chứa thư mục output cũ và đặt `RESUME_DIR_OVERRIDE` tới thư mục đó.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/experiment1_full41_qwen2_5_7b")
RESUME_DIR_OVERRIDE = ""  # Optional attached prior output directory.
if RESUME_DIR_OVERRIDE and not OUTPUT_DIR.exists():
    shutil.copytree(Path(RESUME_DIR_OVERRIDE), OUTPUT_DIR)
command = [
    sys.executable, str(WSDM_DIR / "approved_detection_experiment.py"),
    "extract-proposed", "--inputs", str(WORKING_INPUT),
    "--model", MODEL_SPEC, "--out", str(OUTPUT_DIR),
    "--max-extraction-retries", str(MAX_RETRIES),
    "--expected-items", str(EXPECTED_ITEMS),
    "--model-artifact-digest", MODEL_DIGEST, "--resume",
]
print("Running inference-only command:", " ".join(command))
subprocess.run(command, cwd=str(REPO_DIR), check=True)
run_manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
assert run_manifest["completed"] is True
assert run_manifest["n_processed"] == EXPECTED_ITEMS
assert run_manifest["implementation_sha"] == EXECUTION_COMMIT
assert run_manifest["input_sha256"] == INPUT_SHA256
assert run_manifest["run_configuration"]["model_artifact_digest"] == MODEL_DIGEST
print(json.dumps(run_manifest, indent=2, ensure_ascii=False))
print("Full inference checkpoint: COMPLETE")

## Checkpoint 4 — inspect và lưu inference bundle trước gold

Kiểm tra coverage, attempts, failures và evidence. Sau cell này hãy **Save Version** để giữ raw inference output trước khi attach/read gold.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
attempts = read_jsonl(OUTPUT_DIR / "extraction_attempts.jsonl")
errors = read_jsonl(OUTPUT_DIR / "extraction_errors.jsonl")
status = pd.read_csv(OUTPUT_DIR / "item_status.csv", keep_default_na=False)
evidence = pd.read_csv(OUTPUT_DIR / "item_level_predictions_evidence.csv", keep_default_na=False)
predictions = pd.read_csv(OUTPUT_DIR / "proposed_predictions.csv", keep_default_na=False)
assert len(status) == EXPECTED_ITEMS and status.argument_pair_id.nunique() == EXPECTED_ITEMS
assert set(status.argument_pair_id) == set(inputs.argument_pair_id)
assert not set(predictions.argument_pair_id) & {x["argument_pair_id"] for x in errors}
print(status.status.value_counts())
print(f"attempts={len(attempts)}, retries={sum(int(x['attempt']) > 1 for x in attempts)}, failures={len(errors)}")
display(evidence.head(10))
if errors:
    display(pd.DataFrame(errors))
INFERENCE_ZIP = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))
print(f"Inference ZIP: {INFERENCE_ZIP}")
print("Không có gold nào được đọc; chưa tính AUROC/AP. Hãy Save Version tại đây.")

## Checkpoint 5 — post-inference gold evaluation

Chỉ chạy sau khi inference hoàn tất và đã Save Version. Attach một **dataset riêng** chứa `gold_labels.csv`; file phải có SHA-256 đã audit. Gold chỉ được join trong evaluator, không quay lại prompt/cache/checkpoints.

In [ ]:
GOLD_FILENAME = "gold_labels.csv"
GOLD_SHA256 = "49029491CC7CD010D144174313E48D3F185E65DF2C449FB102419FFF9E0BFEDB"
gold_matches = sorted(Path("/kaggle/input").rglob(GOLD_FILENAME))
if len(gold_matches) != 1:
    raise RuntimeError(f"Attach đúng một separate gold dataset sau inference; tìm thấy: {gold_matches}")
GOLD_PATH = gold_matches[0]
assert hashlib.sha256(GOLD_PATH.read_bytes()).hexdigest().upper() == GOLD_SHA256
gold = pd.read_csv(GOLD_PATH, keep_default_na=False, encoding="utf-8-sig")
assert len(gold) == 41 and gold.argument_pair_id.nunique() == 41
assert set(gold.argument_pair_id) == set(inputs.argument_pair_id)
EVAL_DIR = Path("/kaggle/working/experiment1_full41_evaluation")
if EVAL_DIR.exists():
    raise FileExistsError(f"Evaluation output already exists: {EVAL_DIR}")
eval_command = [
    sys.executable, str(WSDM_DIR / "approved_detection_experiment.py"),
    "evaluate-proposed",
    "--predictions", str(OUTPUT_DIR / "proposed_predictions.csv"),
    "--errors", str(OUTPUT_DIR / "extraction_errors.jsonl"),
    "--attempts", str(OUTPUT_DIR / "extraction_attempts.jsonl"),
    "--gold", str(GOLD_PATH),
    "--evidence", str(OUTPUT_DIR / "item_level_predictions_evidence.csv"),
    "--baseline-predictions", str(REPO_DIR / "wsdm/results/experiment1_approved_detection_20261117T005823/baselines/baseline_predictions.csv"),
    "--out", str(EVAL_DIR),
]
print("Running post-inference evaluation:", " ".join(eval_command))
subprocess.run(eval_command, cwd=str(REPO_DIR), check=True)
summary = json.loads((EVAL_DIR / "proposed_summary.json").read_text(encoding="utf-8"))
display(pd.read_csv(EVAL_DIR / "baseline_proposed_comparison.csv"))
display(pd.read_csv(EVAL_DIR / "ablation_metrics.csv"))
display(pd.read_csv(EVAL_DIR / "missing_score_sensitivity.csv"))
display(pd.read_csv(EVAL_DIR / "error_analysis.csv"))
print((EVAL_DIR / "experiment1_full_report.md").read_text(encoding="utf-8"))
EVALUATION_ZIP = Path(shutil.make_archive(str(EVAL_DIR), "zip", root_dir=EVAL_DIR))
print(f"Evaluation ZIP: {EVALUATION_ZIP}")

## Reporting constraints

- Luôn báo extraction coverage cùng AUROC/AP.
- Failed extractions giữ missing, không gán score 0.
- Năm diagnostic development items nằm trong 41 units.
- Đây là `human_segmented` approved pilot, không phải held-out test hay generalization result.
- Không chọn threshold từ pilot này.